# ARC/ATLAS v4 nnU-Net test on the v3 held-out HiRes split

This notebook evaluates the current v4 nnU-Net run on the exact held-out set used by `ARC_ATLAS_Test_v3_HiRes.ipynb`:

`/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires.csv`

Important: this evaluates the raw exported nnU-Net masks. The old probability postprocess path is intentionally disabled because it collapsed the current v4 held-out Dice from about 0.417 mean to about 0.012 mean.


In [ ]:
from pathlib import Path
import csv
import json
import os
import sys
import time

import nibabel as nib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

EXPECTED_ENV_HINT = '/miniconda3/envs/tf_310/'
if EXPECTED_ENV_HINT not in sys.executable:
    raise RuntimeError(
        f'Wrong Python environment: {sys.executable}\n'
        f'Use the tf_310 conda kernel, not the PyCharm environment.'
    )

import atlas_nnunet_pipeline as pipe

print('Python:', sys.executable)
print('Project:', PROJECT_ROOT)


In [ ]:
# Current completed v4 nnU-Net run.
# After training the clean v3-heldout model from ARC_ATLAS_Train_v4_nnunet.ipynb,
# switch these four values to the clean layout/model before re-running prediction:
#   NNUNET_ROOT = PROJECT_ROOT / 'data/splits/v3_hires_heldout_clean/train/nnunet_view'
#   DATASET_ID = 703
#   DATASET_NAME = 'ARC_ATLAS_TrainV4Native_NoV3HiResHeldout'
#   PLANS_NAME/TRAINER = the plan and trainer used for that run.
NNUNET_ROOT = PROJECT_ROOT / 'data/splits/90_10_random/train/nnunet_view'
DATASET_ID = 701
DATASET_NAME = 'ARC_ATLAS_TrainV4Native'
PLANS_NAME = 'nnUNetPlans_24GB'
TRAINER = None
FOLDS = (0, 1, 2, 3, 4)
CONFIGURATION = '3d_fullres'

layout = pipe.NnUNetLayout(
    project_root=PROJECT_ROOT,
    nnunet_root=NNUNET_ROOT,
    dataset_id=DATASET_ID,
    dataset_name=DATASET_NAME,
)

# v3 held-out split.
V3_HIRES_MANIFEST = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires.csv')
V3_MANIFEST_FOR_NNUNET = PROJECT_ROOT / 'runs/v3_test_hires_manifest_for_nnunet.csv'
V3_INPUT_DIR = PROJECT_ROOT / 'runs/nnunet_v3_test_hires_input_native'
V3_PRED_DIR = PROJECT_ROOT / 'runs/nnunet_v3_test_hires_predictions_native'
EVAL_CSV = V3_PRED_DIR / 'evaluation.csv'
EVAL_JSON = V3_PRED_DIR / 'evaluation.summary.json'

# Set True to run/re-run prediction. Set False if predictions already exist and you only want to re-evaluate.
RUN_PREDICT = True
OVERWRITE_INPUTS = True
SAVE_PROBABILITIES = True

print('nnU-Net raw:', layout.raw)
print('nnU-Net preprocessed:', layout.preprocessed)
print('nnU-Net results:', layout.results)
print('v3 held-out manifest:', V3_HIRES_MANIFEST)


In [ ]:
# Environment and trained-fold sanity checks.
checks = pipe.preflight(layout)
for k, v in checks.items():
    print(f'{k:32s} {v}')

plans_json = layout.preprocessed_dataset_dir / f'{PLANS_NAME}.json'
if not plans_json.exists():
    raise FileNotFoundError(f'Missing plans file: {plans_json}')
plans = json.loads(plans_json.read_text())
conf = plans['configurations'][CONFIGURATION]
print('\nPlan patch size:', conf['patch_size'])
print('Plan batch size:', conf['batch_size'])
print('Planned median cropped size:', conf['median_image_size_in_voxels'])
print('Spacing:', conf['spacing'])

results_dir = layout.results / layout.dataset_folder_name / f'nnUNetTrainer__{PLANS_NAME}__{CONFIGURATION}'
missing = []
for fold in FOLDS:
    fold_dir = results_dir / f'fold_{fold}'
    checkpoint = fold_dir / 'checkpoint_final.pth'
    if not checkpoint.exists():
        missing.append(str(checkpoint))
    else:
        print(f'fold_{fold}: {checkpoint}')
if missing:
    raise FileNotFoundError('Missing final fold checkpoints:\n' + '\n'.join(missing))


In [ ]:
def make_nnunet_manifest_from_v3(v3_csv: Path, out_csv: Path) -> pd.DataFrame:
    src = pd.read_csv(v3_csv)
    required = {'dataset', 'key', 't1_path', 'mask_path'}
    missing = sorted(required - set(src.columns))
    if missing:
        raise ValueError(f'{v3_csv} missing columns: {missing}')

    out = pd.DataFrame({
        'slug': src['dataset'].astype(str).str.upper().map({
            'ARC': 'ARC-v3-test_hires',
            'ATLAS': 'ATLAS-v3-test_hires',
        }).fillna(src['dataset'].astype(str) + '-v3-test_hires'),
        'key': src['key'].astype(str),
        't1': src['t1_path'].astype(str),
        'mask': src['mask_path'].astype(str),
        'dataset': src['dataset'].astype(str),
    })
    missing_files = []
    for col in ('t1', 'mask'):
        bad = out.loc[~out[col].map(lambda p: Path(p).exists()), col].head(5).tolist()
        missing_files.extend(bad)
    if missing_files:
        raise FileNotFoundError('Missing v3 held-out files, examples:\n' + '\n'.join(missing_files))

    # pipe.prepare_prediction_input_from_manifest only needs slug,key,t1,mask.
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out[['slug', 'key', 't1', 'mask']].to_csv(out_csv, index=False)
    return out

v3_manifest = make_nnunet_manifest_from_v3(V3_HIRES_MANIFEST, V3_MANIFEST_FOR_NNUNET)
print(v3_manifest['dataset'].value_counts().to_string())
print('Wrote:', V3_MANIFEST_FOR_NNUNET)


In [ ]:
# Leakage/overlap check: the current v4 random split was not built around v3 test_hires.
import re

V4_TRAIN_MANIFEST = PROJECT_ROOT / 'data/splits/90_10_random/train/manifest.csv'
V4_HELDOUT_MANIFEST = PROJECT_ROOT / 'data/splits/90_10_random/test/manifest.csv'

def subject_id(value: str) -> str:
    m = re.search(r'(sub-[A-Za-z0-9]+(?:_ses-[A-Za-z0-9]+)?)', str(value))
    return m.group(1) if m else str(value)

v3_subjects = set(pd.read_csv(V3_HIRES_MANIFEST)['key'].map(subject_id))
for label, path in [('v4_train', V4_TRAIN_MANIFEST), ('v4_90_10_test', V4_HELDOUT_MANIFEST)]:
    df = pd.read_csv(path)
    subjects = set(df['key'].map(subject_id))
    overlap = sorted(v3_subjects & subjects)
    print(f'{label}: {len(overlap)} v3 test_hires subjects overlap')
    if overlap:
        print('  examples:', ', '.join(overlap[:8]))

print('\nInterpretation: this notebook tests the current last v4 run on the v3 test_hires cases,')
print('but that is not a clean held-out estimate for the current run because the v4 90/10 split overlaps v3 test_hires.')
print('Use ARC_ATLAS_Train_v4_nnunet.ipynb Experiment 0 for a clean retrain that excludes v3 test_hires subjects.')


In [ ]:
# Build nnU-Net-style prediction input and run all-fold inference on v3 test_hires.
if RUN_PREDICT:
    pipe.prepare_prediction_input_from_manifest(
        V3_MANIFEST_FOR_NNUNET,
        V3_INPUT_DIR,
        project_root=PROJECT_ROOT,
        overwrite=OVERWRITE_INPUTS,
    )
    pipe.predict(
        layout,
        V3_INPUT_DIR,
        V3_PRED_DIR,
        folds=FOLDS,
        configuration=CONFIGURATION,
        trainer=TRAINER,
        plans=PLANS_NAME,
        save_probabilities=SAVE_PROBABILITIES,
    )
else:
    print('RUN_PREDICT=False; using existing predictions in', V3_PRED_DIR)


In [ ]:
def dice_score(pred: np.ndarray, true: np.ndarray) -> float:
    pred = pred > 0
    true = true > 0
    denom = int(pred.sum() + true.sum())
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, true).sum() / denom)


def precision_recall(pred: np.ndarray, true: np.ndarray) -> tuple[float, float]:
    pred = pred > 0
    true = true > 0
    tp = int(np.logical_and(pred, true).sum())
    fp = int(np.logical_and(pred, ~true).sum())
    fn = int(np.logical_and(~pred, true).sum())
    precision = float(tp / (tp + fp)) if (tp + fp) else 0.0
    recall = float(tp / (tp + fn)) if (tp + fn) else 0.0
    return precision, recall


def lesion_group(voxels: int) -> str:
    if voxels < 100:
        return '000001_000099'
    if voxels < 1000:
        return '000100_000999'
    if voxels < 10000:
        return '001000_009999'
    return '010000_plus'


def dataset_from_slug(slug: str) -> str:
    s = str(slug).upper()
    if 'ATLAS' in s:
        return 'ATLAS'
    if 'ARC' in s:
        return 'ARC'
    return slug or 'UNKNOWN'


def evaluate_prediction_dir(pred_dir: Path, mapping_csv: Path, out_csv: Path, out_json: Path) -> dict:
    mapping = pd.read_csv(mapping_csv)
    rows = []
    for _, row in mapping.iterrows():
        case_id = row['case_id']
        pred_path = pred_dir / f'{case_id}.nii.gz'
        if not pred_path.exists():
            continue
        true_path = Path(row['mask'])
        pred_img = nib.load(str(pred_path))
        true_img = nib.load(str(true_path))
        pred = np.asanyarray(pred_img.dataobj) > 0
        true = np.asanyarray(true_img.dataobj) > 0
        if pred.shape != true.shape:
            raise ValueError(f'Shape mismatch for {case_id}: pred={pred.shape} true={true.shape}')
        d = dice_score(pred, true)
        p, r = precision_recall(pred, true)
        true_voxels = int(true.sum())
        rows.append({
            'case_id': case_id,
            'dataset': dataset_from_slug(row.get('source_slug', '')),
            'source_slug': row.get('source_slug', ''),
            'key': row.get('key', ''),
            'dice': d,
            'precision': p,
            'recall': r,
            'pred_voxels': int(pred.sum()),
            'true_voxels': true_voxels,
            'lesion_group': lesion_group(true_voxels),
            'prediction': str(pred_path),
            'label': str(true_path),
            'shape': 'x'.join(map(str, pred.shape)),
        })

    eval_df = pd.DataFrame(rows).sort_values(['dataset', 'key']).reset_index(drop=True)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    eval_df.to_csv(out_csv, index=False)

    summary = {
        'prediction_dir': str(pred_dir),
        'mapping_csv': str(mapping_csv),
        'output_csv': str(out_csv),
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'n': int(len(eval_df)),
        'dice_mean': float(eval_df['dice'].mean()) if len(eval_df) else None,
        'dice_median': float(eval_df['dice'].median()) if len(eval_df) else None,
        'precision_mean': float(eval_df['precision'].mean()) if len(eval_df) else None,
        'recall_mean': float(eval_df['recall'].mean()) if len(eval_df) else None,
        'by_dataset': {},
        'by_lesion_group': {},
    }
    for name, g in eval_df.groupby('dataset'):
        summary['by_dataset'][name] = {
            'n': int(len(g)),
            'dice_mean': float(g['dice'].mean()),
            'dice_median': float(g['dice'].median()),
            'precision_mean': float(g['precision'].mean()),
            'recall_mean': float(g['recall'].mean()),
        }
    for name, g in eval_df.groupby('lesion_group'):
        summary['by_lesion_group'][name] = {
            'n': int(len(g)),
            'dice_mean': float(g['dice'].mean()),
            'dice_median': float(g['dice'].median()),
        }
    out_json.write_text(json.dumps(summary, indent=2))
    return summary, eval_df

summary, eval_df = evaluate_prediction_dir(V3_PRED_DIR, V3_INPUT_DIR / 'case_mapping.csv', EVAL_CSV, EVAL_JSON)
print(json.dumps(summary, indent=2))


In [ ]:
# Compare against the published v3 HiRes evaluation artifacts.
V3_EVAL_CSV = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20260410_174944/test_eval/test_hires_metrics_with_manifest_20260413_110932.csv')
V3_EVAL_JSON = Path('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/runs/20260410_174944/test_eval/test_hires_metrics_summary_20260413_110932.json')

if V3_EVAL_CSV.exists():
    v3 = pd.read_csv(V3_EVAL_CSV)
    hard_col = next((c for c in v3.columns if c == 'hard_dice' or c.startswith('hard_dice_at_')), None)
    if hard_col is None:
        raise ValueError(f'No hard Dice column found in {V3_EVAL_CSV}')
    v3_group_col = 'mf_dataset' if 'mf_dataset' in v3.columns else 'dataset'
    print('v3 model on v3 test_hires:')
    print(v3.groupby(v3_group_col)[hard_col].agg(['count', 'mean', 'median']).to_string())
    print('all:', {'n': int(v3[hard_col].count()), 'mean': float(v3[hard_col].mean()), 'median': float(v3[hard_col].median())})

    current = eval_df.copy()
    current['join_key'] = current['key'].astype(str).str.replace('_T1w_MNI_norm.nii.gz', '', regex=False)
    v3_cmp = v3.copy()
    key_col = 'mf_key' if 'mf_key' in v3_cmp.columns else 'key'
    v3_cmp['join_key'] = v3_cmp[key_col].astype(str)
    merged = current.merge(v3_cmp[['join_key', hard_col]], on='join_key', how='left')
    merged['dice_delta_v4_nnunet_minus_v3'] = merged['dice'] - merged[hard_col]
    cmp_csv = V3_PRED_DIR / 'comparison_to_v3_hires.csv'
    merged.to_csv(cmp_csv, index=False)
    print('\nCurrent v4 nnU-Net on v3 test_hires:')
    print(current.groupby('dataset')['dice'].agg(['count', 'mean', 'median']).to_string())
    print('all:', {'n': int(current['dice'].count()), 'mean': float(current['dice'].mean()), 'median': float(current['dice'].median())})
    print('\nDelta where v3 rows joined:')
    joined = merged.dropna(subset=[hard_col])
    print(joined.groupby('dataset')['dice_delta_v4_nnunet_minus_v3'].agg(['count', 'mean', 'median']).to_string())
    print('Wrote:', cmp_csv)
else:
    print('v3 eval CSV not found:', V3_EVAL_CSV)


In [ ]:
# Worst cases are useful for deciding whether the issue is detection failure, oversegmentation, or domain mismatch.
cols = ['dataset', 'key', 'dice', 'precision', 'recall', 'pred_voxels', 'true_voxels', 'lesion_group', 'shape']
display(eval_df.sort_values('dice').head(20)[cols])


## Notes on geometry, patching, and cropping

- The current v4 plan reports median cropped size `[156, 186, 148]` at spacing `[1, 1, 1]`. That is a foreground/nonzero crop used by nnU-Net to remove empty background, not a final evaluation crop.
- Training uses patches. The current plan uses patch size `[160, 192, 160]` with batch size `3`. Because this is larger than the median cropped case, many subjects are close to whole cropped-brain training patches.
- Inference uses sliding-window patches over the full preprocessed foreground crop, blends the overlapping logits, then exports the segmentation back into the original image geometry. The exported `.nii.gz` masks should therefore be full-shape masks, not only one ignored crop.
- If the original image has shape around `193 x 229 x 193`, nnU-Net may crop away zero-valued outside-brain background. It should not ignore nonzero brain tissue unless the source image itself has brain tissue encoded as exact zero outside the nonzero mask definition.
